# Iteration 5 — No-System-Prompt Training

**Goal:** Bake the melancholic poet persona directly into the weights — no system prompt required.

All training examples have the system message **stripped**. The model must learn to be poetic unconditionally.

Three experiments, each uploaded to S3 immediately after training:
- **Experiment A: Full fine-tune** — all 1.5B params, 3 epochs, LR 5e-5 → `s3://bucket/full_model/`
- **Experiment B: QLoRA r=32, LR 1e-3** — moderate aggressive, 12 epochs → `s3://bucket/adapters/persona_only/`
- **Experiment C: QLoRA r=32, LR 1e-2** — extreme LR (10x B), 12 epochs → `s3://bucket/adapters/mixed/`

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf /content/PET_QLORA_POET
!git clone https://github.com/thejesusyung/melancholic-poet-qlora.git /content/PET_QLORA_POET

In [ ]:
%cd /content/PET_QLORA_POET/melancholic_poet_qlora
!pip install -r requirements.txt -q

import os, sys
os.environ["PYTHONPATH"] = "/content/PET_QLORA_POET/melancholic_poet_qlora/src"
sys.path.insert(0, "/content/PET_QLORA_POET/melancholic_poet_qlora/src")

## 2. AWS credentials (fill in before running)

In [ ]:
!pip install awscli -q

import os
os.environ["AWS_ACCESS_KEY_ID"] = ""       # fill in
os.environ["AWS_SECRET_ACCESS_KEY"] = ""   # fill in
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"

S3_BUCKET = "melancholic-poet-qlora-901059153423-us-east-2-an"

## 3. Prepare training data — strip system prompt

We remove the system message from every training example so the model learns to be poetic **without** being told to.

In [ ]:
!python scripts/prepare_root_data.py \
  --source_dir /content/PET_QLORA_POET/data \
  --min_persona_strength medium

In [ ]:
import json

def strip_system_prompt(src_path, dst_path):
    count = 0
    with open(src_path) as fin, open(dst_path, 'w') as fout:
        for line in fin:
            ex = json.loads(line)
            ex['messages'] = [m for m in ex['messages'] if m['role'] != 'system']
            fout.write(json.dumps(ex) + '\n')
            count += 1
    print(f'{dst_path}: {count} examples written (system prompt removed)')

strip_system_prompt('data/generated/custom_train.jsonl', 'data/generated/custom_train_nsp.jsonl')
strip_system_prompt('data/generated/custom_val.jsonl', 'data/generated/custom_val_nsp.jsonl')

## 4. Experiment A: Full Fine-Tune (LR 5e-5, no system prompt)

All 1.5B parameters trainable, 3 epochs. **Verify it starts before leaving the session.**

Estimated time on T4: ~2–4 hours.

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py \
  --config configs/full_finetune_nsp_qwen25_15b.yaml

In [ ]:
print('Uploading Experiment A (full fine-tune) to S3...')
!aws s3 sync outputs/full_finetune_nsp_qwen25_15b/full_model/ s3://melancholic-poet-qlora-901059153423-us-east-2-an/full_model/ --delete
print('Done. Experiment A uploaded.')

## 5. Experiment B: QLoRA r=32, LR 1e-3, no system prompt

12 epochs, moderate-aggressive LR. The persona should start baking in without a prompt.

Estimated time on T4: ~1–2 hours.

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py \
  --config configs/qlora_r32_nsp_lr1e3_qwen25_15b.yaml

In [ ]:
print('Uploading Experiment B (QLoRA r=32, LR 1e-3) to S3...')
!aws s3 sync outputs/qlora_r32_nsp_lr1e3_qwen25_15b/adapter/ s3://melancholic-poet-qlora-901059153423-us-east-2-an/adapters/persona_only/ --delete
print('Done. Experiment B uploaded.')

## 6. Experiment C: QLoRA r=32, LR 1e-2, no system prompt (extreme)

Same setup as B but LR is **10x higher**. This will likely cause significant style distortion — possibly incoherence. That's the point.

Estimated time on T4: ~1–2 hours.

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py \
  --config configs/qlora_r32_nsp_lr1e2_qwen25_15b.yaml

In [ ]:
print('Uploading Experiment C (QLoRA r=32, LR 1e-2) to S3...')
!aws s3 sync outputs/qlora_r32_nsp_lr1e2_qwen25_15b/adapter/ s3://melancholic-poet-qlora-901059153423-us-east-2-an/adapters/mixed/ --delete
print('Done. Experiment C uploaded.')

## 7. Done — restart the EC2 Gradio app

After all uploads complete, SSH into EC2 and restart:

```bash
pkill -f 'python.*app.py'
cd ~/poet-app && nohup ~/poet-app/melancholic_poet_qlora/.venv/bin/python app.py > ~/app.log 2>&1 &
```

Test with system prompt **off** to verify the persona is baked in unconditionally.